# Lesson 19: Causal Inference in Machine Learning

## Opening Story: Recommendation Systems

Netflix recommends movies based on what you've watched. But is this causal? If you watch a thriller because Netflix recommended it, and then you get more thriller recommendations, is Netflix actually causing your preferences?

Machine learning is excellent at prediction but struggles with causation. Causal inference in ML bridges this gap, enabling algorithms that understand not just what happened, but why.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Explain why ML needs causal inference
2. Implement causal forests
3. Use double machine learning
4. Apply meta-learners for treatment effect estimation
5. Recognize limitations of standard ML for causal questions

---

## 19.1 Why ML Needs Causation

### Prediction vs. Causation

- **Prediction**: What will happen given observed features?
- **Causation**: What will happen if we intervene?

Standard ML optimizes prediction accuracy. Causal ML optimizes treatment effect estimation.

### The Problem with Correlations

ML algorithms learn correlations, which can be:
- Spurious (due to confounding)
- Reverse (effect predicted as cause)
- Non-causal (due to selection)

---

## 19.2 Causal Forests

In [ ]:
import numpy as np
import pandas as pd
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor

np.random.seed(42)
n = 2000
p = 10

# Generate data
X = np.random.normal(0, 1, (n, p))
T = np.random.binomial(1, 0.5, n)

# Heterogeneous treatment effect
true_effect = 2 * X[:, 0] + X[:, 1] ** 2
Y = true_effect * T + X @ np.ones(p) + np.random.normal(0, 1, n)

# Fit causal forest
causal_forest = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=100, random_state=42),
    model_t=RandomForestRegressor(n_estimators=100, random_state=42),
    n_estimators=100,
    random_state=42
)

causal_forest.fit(Y, T, X=X)

# Predict treatment effects
te_pred = causal_forest.effect(X)

# Evaluate
correlation = np.corrcoef(true_effect, te_pred)[0, 1]
mse = np.mean((true_effect - te_pred) ** 2)

print(f"Correlation between true and predicted effects: {correlation:.3f}")
print(f"MSE: {mse:.3f}")

---

## 19.3 Double Machine Learning

In [ ]:
from econml.dml import LinearDML

# Double ML for average treatment effect
dml = LinearDML(
    model_y=RandomForestRegressor(n_estimators=100, random_state=42),
    model_t=RandomForestRegressor(n_estimators=100, random_state=42),
    random_state=42
)

dml.fit(Y, T, X=X)

# Average treatment effect
ate = dml.ate(X)
ate_interval = dml.ate_interval(X, alpha=0.05)

print(f"ATE estimate: {ate:.3f}")
print(f"95% CI: [{ate_interval[0]:.3f}, {ate_interval[1]:.3f}]")
print(f"True ATE: {true_effect.mean():.3f}")

---

## 19.4 Meta-Learners

In [ ]:
from econml.metalearners import TLearner, SLearner, XLearner

# S-Learner
s_learner = SLearner(
    overall_model=RandomForestRegressor(n_estimators=100, random_state=42)
)
s_learner.fit(Y, T, X=X)
te_s = s_learner.effect(X)

# T-Learner
t_learner = TLearner(
    models=RandomForestRegressor(n_estimators=100, random_state=42)
)
t_learner.fit(Y, T, X=X)
te_t = t_learner.effect(X)

# X-Learner
x_learner = XLearner(
    models=RandomForestRegressor(n_estimators=100, random_state=42)
)
x_learner.fit(Y, T, X=X)
te_x = x_learner.effect(X)

print("S-Learner correlation:", np.corrcoef(true_effect, te_s)[0, 1].round(3))
print("T-Learner correlation:", np.corrcoef(true_effect, te_t)[0, 1].round(3))
print("X-Learner correlation:", np.corrcoef(true_effect, te_x)[0, 1].round(3))

---

## 19.5 Common Mistakes

1. **Using ML for causal inference directly**: Standard ML gives biased treatment effect estimates
2. **Ignoring confounding**: ML doesn't handle confounding by default
3. **Overfitting**: Use cross-fitting and sample splitting
4. **Wrong evaluation metrics**: Use causal metrics, not prediction metrics

---

## 19.6 Knowledge Check

### Multiple Choice

1. **Standard ML:**
   A) Handles causation naturally
   B) Optimizes prediction accuracy
   C) Estimates treatment effects
   D) All of the above

2. **Causal forests estimate:**
   A) Average treatment effects
   B) Conditional average treatment effects
   C) Total effects
   D) Direct effects

3. **Double ML:**
   A) Requires two datasets
   B) Uses cross-fitting to reduce bias
   C) Doubles the sample size
   D) Is always better than single ML

4. **Meta-learners:**
   A) Are always identical
   B) Handle heterogeneous effects
   C) Don't need any assumptions
   D) Are only for experiments

5. **Cross-fitting is used to:**
   A) Increase overfitting
   B) Reduce overfitting
   C) Speed up computation
   D) Increase variance

### Short Answer

6. **Why can't we use standard ML for causal inference?**

7. **What is the role of cross-fitting in causal ML?**

8. **How do meta-learners differ from each other?**

9. **When would you use causal forests vs. double ML?**

10. **What are the limitations of causal ML methods?**

---

## 19.7 Summary

1. **ML needs causation** for treatment effect estimation
2. **Causal forests** estimate heterogeneous effects
3. **Double ML** uses cross-fitting for valid inference
4. **Meta-learners** provide flexible effect estimation
5. **Cross-fitting** is essential for validity

---

## 19.8 Further Reading

- Athey, S. & Imbens, G.W. (2019). "Machine Learning Methods That Economists Should Know About." *Annual Review of Economics*.
- Chernozhukov, V. et al. (2018). "Double/Debiased Machine Learning." *Econometrics Journal*.